# 05 Final Load Prep

Prepare Formula 1 dataset for Tableau by merging results, races, drivers, and constructors. This notebook generates the final `tableau_ready_dataset.csv` containing all necessary dimensions and KPIs.

In [ ]:
from pathlib import Path
import pandas as pd

# Setup Paths
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
RAW_DATA = PROJECT_ROOT / 'data/raw'
PROCESSED_DATA = PROJECT_ROOT / 'data/processed'
TABLEAU_READY_PATH = PROCESSED_DATA / 'tableau_ready_dataset.csv'

print(f"Project root: {PROJECT_ROOT}")

### 1. Load Core Datasets
Merge the primary data to create a rich analytical table.

In [ ]:
# Load files
results = pd.read_csv(PROCESSED_DATA / 'Cleaned Results.csv')
races = pd.read_csv(PROCESSED_DATA / 'Cleaned Races.csv')
drivers = pd.read_csv(PROCESSED_DATA / 'Clean Drivers Data.csv')
constructors = pd.read_csv(PROCESSED_DATA / 'Constructors Clean Data.csv')
status = pd.read_csv(PROCESSED_DATA / 'Cleaned Status.csv')
circuits = pd.read_csv(PROCESSED_DATA / 'Clean Circuits Data.csv')
circuits_raw = pd.read_csv(RAW_DATA / 'circuits.csv', encoding='latin1')[['circuitId', 'alt']]
circuits = circuits.merge(circuits_raw, on='circuitId', how='left')

# Clean up driver names
drivers['driver_name'] = drivers['forename'] + ' ' + drivers['surname']

# Merge results with races
df = results.merge(races[['raceId', 'year', 'name', 'date', 'circuitId']], on='raceId', how='left')
df.rename(columns={'name': 'race_name'}, inplace=True)

# Merge with drivers
df = df.merge(drivers[['driverId', 'driverRef', 'driver_name', 'nationality']], on='driverId', how='left')
df.rename(columns={'nationality': 'driver_nationality'}, inplace=True)

# Merge with constructors
df = df.merge(constructors[['constructorId', 'constructorRef', 'name', 'nationality']], on='constructorId', how='left')
df.rename(columns={'name': 'team_name', 'nationality': 'team_nationality'}, inplace=True)

# Merge with status (DNF reasons)
df = df.merge(status[['statusId', 'status']], on='statusId', how='left')

# Merge with circuits
df = df.merge(circuits[['circuitId', 'location', 'country', 'lat', 'lng', 'alt']], on='circuitId', how='left')

print(f"Merged Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### 2. Compute KPIs
Add fields for easy Tableau calculations.

In [ ]:
# 1. Win Indicator
df['is_win'] = (df['positionOrder'] == 1).astype(int)

# 2. Podium Indicator (Top 3)
df['is_podium'] = (df['positionOrder'] <= 3).astype(int)

# 3. Points Finish (Top 10)
df['is_points_finish'] = (df['points'] > 0).astype(int)

# 4. Grid to Finish delta
df['grid_delta'] = df['grid'] - df['positionOrder']

df[['driver_name', 'year', 'points', 'is_win', 'is_podium']].head()

### 3. Final Export
Save the file for use in Tableau dashboards.

In [ ]:
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)
df.to_csv(TABLEAU_READY_PATH, index=False)
print(f"SUCCESS: Tableau-ready dataset saved to {TABLEAU_READY_PATH}")